# This Section Concerns all mentioned Datasets with Random Forest

# Imports

In [14]:
# Standard library
import os

# Data handling
import numpy as np
import pandas as pd
from scipy import sparse

# GPU libraries (optional for cuML models)
import cupy as cp
from cuml.ensemble import RandomForestClassifier


# ML utilities
from cuml.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score


# Visualization
import seaborn as sns
import matplotlib.pyplot as plt


In [15]:

def to_gpu_dense(X):
    """
    Converts input matrix X into a dense CuPy array safely.
    - If X is SciPy CSR/CSC/COO sparse → convert to NumPy dense → CuPy
    - If X is pandas DataFrame → convert to NumPy → CuPy
    - If X is NumPy array → CuPy
    - Never return nested object-arrays
    """

    # Case 1: SciPy sparse (CSR, CSC, COO)
    if sparse.issparse(X):
        # Convert sparse → dense NumPy → CuPy
        X_np = X.toarray().astype(np.float32)
        return cp.asarray(X_np)

    # Case 2: Pandas DataFrame
    if hasattr(X, "values"):
        return cp.asarray(X.values.astype(np.float32))

    # Case 3: NumPy array
    if isinstance(X, np.ndarray):
        return cp.asarray(X.astype(np.float32))

    # If it's already CuPy
    if isinstance(X, cp.ndarray):
        return X

    raise TypeError(f"Unsupported type passed to to_gpu_dense(): {type(X)}")

def load_split(data, split, base_path="../data/splits/"):
    """
    Loads X_train, X_test, y_train, y_test for a given dataset + split.
    Automatically detects whether features are stored as sparse (.npz)
    or dense (.csv).

    Example:
        X_train, X_test, y_train, y_test = load_split("cup98", "7030")
    """

    path = os.path.join(base_path, split)

    # ---- Load X_train ----
    npz_path = os.path.join(path, f"X_train_{data}.npz")
    csv_path = os.path.join(path, f"X_train_{data}.csv")

    if os.path.exists(npz_path):
        X_train = sparse.load_npz(npz_path)
    else:
        X_train = pd.read_csv(csv_path)

    # ---- Load X_test ----
    npz_path = os.path.join(path, f"X_test_{data}.npz")
    csv_path = os.path.join(path, f"X_test_{data}.csv")

    if os.path.exists(npz_path):
        X_test = sparse.load_npz(npz_path)
    else:
        X_test = pd.read_csv(csv_path)

    # ---- Load labels ----
    y_train = pd.read_csv(os.path.join(path, f"y_train_{data}.csv"))
    y_test  = pd.read_csv(os.path.join(path, f"y_test_{data}.csv"))

    # Convert DataFrames → Series
    y_train = y_train.iloc[:, 0]
    y_test  = y_test.iloc[:, 0]

    return X_train, X_test, y_train, y_test



def load_all_by_split(datasets, base_path="../data/splits/"):
    split_types = ["7030", "3070", "5050"]
    result = {split: {} for split in split_types}

    for split in split_types:
        print(f"\n=== Loading {split} splits ===")
        for data in datasets:
            print(f"  -> Loading {data}")
            X_train, X_test, y_train, y_test = load_split(data, split, base_path)
            result[split][data] = {
                "X_train": X_train,
                "X_test": X_test,
                "y_train": y_train,
                "y_test": y_test
            }

    return result

# Note on Using Breiman-Style Random Forest mtry Settings in cuML

Breiman’s original Random Forest formulation (Breiman, 2001) recommends selecting the 
number of features to consider at each split (`mtry`) as a function of the input 
dimensionality. In particular, the canonical guideline is:

\[
mtry \in \left\{ \frac{\sqrt{p}}{2},\ \sqrt{p},\ 2\sqrt{p},\ 4\sqrt{p},\ 8\sqrt{p} \right\}
\]

where \( p \) is the total number of features.

In classical CPU-based implementations, `mtry` can be specified as an **integer count**.  
However, **cuML’s RandomForestClassifier does not accept integer `max_features`**, and instead 
requires:

- A **float** in the range `(0, 1]`, representing the *fraction of the total feature space*, or  
- A preset string such as `"sqrt"` or `"log2"`

To implement Breiman-style scaling on GPU while staying faithful to the intended feature 
exposure per split, we convert each `mtry` value into a cuML-compatible fraction:

\[
\text{cuML\_max\_features} = \frac{mtry}{p}
\]

Thus, for a feature space of size \( p \), we compute:

\[
s = \lfloor \sqrt{p} \rfloor
\]

\[
mtry \in \left\{ \left\lfloor \frac{s}{2} \right\rfloor,\ s,\ 2s,\ 4s,\ 8s \right\}
\]

and use:

\[
\text{max\_features} = \frac{mtry}{p}
\]

This mapping preserves the **relative selection pressure** intended by Breiman’s formulation 
while maintaining compatibility with cuML’s API.

For each dataset, we therefore train **5 GPU-accelerated Random Forests**, each using:

- `n_estimators = 500`  
- `max_features = mtry / num_features`  
- `mtry ∈ {s/2, s, 2s, 4s, 8s}`, where `s = floor(sqrt(num_features))`

This configuration provides a more scalable and expressive mtry schedule than the fixed small 
values used in the Breiman–Cutler replication, and it generally yields stronger performance on 
both balanced and high-dimensional datasets while adhering to the theoretical motivation of 
Breiman’s Random Forests.


In [16]:
def RF_Training_Testing(data, dataset_name, split_name, mtry_values=None):
    """
    Trains multiple GPU RF models with Breiman-style mtry settings.
    Returns list of metric-rows for CSV (consistent with LR results).
    """

    all_results = []

    # Convert X to GPU
    X_train = to_gpu_dense(data["X_train"]).astype(cp.float32)
    X_test  = to_gpu_dense(data["X_test"]).astype(cp.float32)

    # Convert labels
    y_train = cp.asarray(data["y_train"]).astype(cp.int32)
    y_test  = cp.asarray(data["y_test"]).astype(cp.int32)

    y_test_cpu = cp.asnumpy(y_test)
    num_features = X_train.shape[1]

    # Default Breiman mtry settings
    if mtry_values is None:
        s = int(np.sqrt(num_features))
        mtry_values = [
            max(1, int(s/2)),
            s,
            2*s,
            4*s,
            8*s
        ]

    print(f"RF feature count = {num_features}")
    print(f"Using mtry values: {mtry_values}")

    # ==========================
    # Loop over mtry candidates
    # ==========================
    for mtry in mtry_values:

        frac = mtry / num_features
        frac = min(max(frac, 1.0 / num_features), 1.0)

        print(f"\nTraining RF with mtry={mtry}, max_features={frac:.5f}")

        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=100,
            max_features=frac,
            n_streams=8
        )

        rf.fit(X_train, y_train)

        preds = rf.predict(X_test)
        preds_cpu = cp.asnumpy(preds)

        # AUC requires probabilities
        probs_cpu = cp.asnumpy(rf.predict_proba(X_test)[:, 1])

        # ---- Metrics ----
        accuracy  = accuracy_score(y_test_cpu, preds_cpu)
        precision = precision_score(y_test_cpu, preds_cpu)
        recall    = recall_score(y_test_cpu, preds_cpu)
        f1        = f1_score(y_test_cpu, preds_cpu)
        auc       = roc_auc_score(y_test_cpu, probs_cpu)

        print(f"Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}, AUC={auc:.4f}")

        # Add a row (same schema as LR)
        all_results.append({
            "dataset": dataset_name,
            "split": split_name,
            "model": "RF",
            "mtry": mtry,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "auc_roc": auc
        })

    return all_results



def evaluate_model(preds_gpu, y_test, title="Model Evaluation"):
    # Convert GPU → CPU
    preds = cp.asnumpy(preds_gpu)
    y_test_cpu = cp.asnumpy(y_test)

    # ======== Accuracy ========
    acc = accuracy_score(y_test_cpu, preds)
    print(f"\n=== {title} ===")
    print(f"Accuracy: {acc:.4f}")

    # ======== Precision / Recall / F1 ========
    print("\nClassification Report:")
    print(classification_report(y_test_cpu, preds))

    # ======== Confusion Matrix ========
    cm = confusion_matrix(y_test_cpu, preds)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
    plt.title(f"Confusion Matrix: {title}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # ======== ROC Curve (Binary only) ========
    if len(cp.unique(y_test)) == 2:  
        try:
            # AUC
            auc = roc_auc_score(y_test_cpu, preds)
            print(f"AUC: {auc:.4f}")

            # ROC curve
            fpr, tpr, _ = roc_curve(y_test_cpu, preds)
            plt.figure(figsize=(6,5))
            plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
            plt.plot([0,1], [0,1], 'k--')
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title(f"ROC Curve: {title}")
            plt.legend()
            plt.show()
        except Exception as e:
            print("Could not compute ROC:", e)
    else:
        print("\nROC/AUC skipped (not binary classification).")

In [17]:
rf_results = []

## Loading Data in

In [18]:
datasets = ["wine", "cup98", "customer"]

all_splits = load_all_by_split(datasets)



=== Loading 7030 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 3070 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 5050 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer


In [19]:

wine_3070 = all_splits["3070"]["wine"]
rf_results += RF_Training_Testing(wine_3070, "wine", "3070")
customer_3070 = all_splits["3070"]["customer"]
rf_results += RF_Training_Testing(customer_3070, "customer", "3070")
cup98_3070 = all_splits["3070"]["cup98"]
rf_results += RF_Training_Testing(cup98_3070, "cup98", "3070")

wine_7030 = all_splits["7030"]["wine"]
rf_results += RF_Training_Testing(wine_7030, "wine", "7030")
customer_7030 = all_splits["7030"]["customer"]
rf_results += RF_Training_Testing(customer_7030, "customer", "7030")
cup98_7030 = all_splits["7030"]["cup98"]
rf_results += RF_Training_Testing(cup98_7030, "cup98", "7030")

wine_5050 = all_splits["5050"]["wine"]
rf_results += RF_Training_Testing(wine_5050, "wine", "5050")
customer_5050 = all_splits["5050"]["customer"]
rf_results += RF_Training_Testing(customer_5050, "customer", "5050")
cup98_5050 = all_splits["5050"]["cup98"]
rf_results += RF_Training_Testing(cup98_5050, "cup98", "5050")


RF feature count = 12
Using mtry values: [1, 3, 6, 12, 24]

Training RF with mtry=1, max_features=0.08333
Accuracy=0.9910, Precision=0.9954, Recall=0.9679, F1=0.9814, AUC=0.9980

Training RF with mtry=3, max_features=0.25000
Accuracy=0.9930, Precision=0.9936, Recall=0.9777, F1=0.9856, AUC=0.9974

Training RF with mtry=6, max_features=0.50000
Accuracy=0.9925, Precision=0.9936, Recall=0.9759, F1=0.9847, AUC=0.9981

Training RF with mtry=12, max_features=1.00000
Accuracy=0.9868, Precision=0.9809, Recall=0.9652, F1=0.9730, AUC=0.9963

Training RF with mtry=24, max_features=1.00000
Accuracy=0.9868, Precision=0.9809, Recall=0.9652, F1=0.9730, AUC=0.9963
RF feature count = 7
Using mtry values: [1, 2, 4, 8, 16]

Training RF with mtry=1, max_features=0.14286
Accuracy=0.6929, Precision=0.7489, Recall=0.6584, F1=0.7008, AUC=0.7545

Training RF with mtry=2, max_features=0.28571
Accuracy=0.6830, Precision=0.7378, Recall=0.6506, F1=0.6915, AUC=0.7559

Training RF with mtry=4, max_features=0.57143
Ac

In [20]:
import pandas as pd

df_rf = pd.DataFrame(rf_results)
df_rf.to_csv("rf_results.csv", index=False)